# Synthetic Operational Risk Data Generation

## Overview
This notebook generates synthetic operational risk data to simulate a Basel-style Enterprise Risk Management (ERM) environment.

While the final project uses the real P-COLD dataset, this synthetic dataset was created during early development to prototype the data pipeline, star schema design, and dashboard structure.

## Objectives
- Simulate operational risk categories, event types, and business units
- Generate synthetic loss events with realistic distributions
- Create mitigation tracking fields and risk attributes
- Export structured dimension and fact tables for testing

## Key Outputs
- Synthetic risk registry
- Synthetic loss event dataset
- Fact and dimension tables for Power BI testing

## Notes
This notebook is retained for completeness and demonstration of the development process but is not used in the final modeling pipeline.

In [1]:
# This notebook generates synthetic operational risk data for early testing.
# The final capstone pipeline uses the real P-COLD dataset, but this notebook is retained
# as an optional synthetic-data benchmark and backup pipeline.

import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta

np.random.seed(42)
fake = Faker()

## ERM Reference Dimension
Basel focused references were chosen to comply with industry standards.

In [2]:
# Define synthetic Basel-style operational risk categories, event types,
# business units, and impact levels used to generate sample ERM records.

risk_categories = {
    "Internal Fraud": ["Unauthorized trading", "Theft of assets"],
    "External Fraud": ["Cyber attack", "Payment fraud"],
    "Employment Practices": ["Discrimination claim", "Workplace injury"],
    "Clients & Products": ["Mis-selling", "Contract breach"],
    "Damage to Assets": ["Fire", "Natural disaster"],
    "Business Disruption": ["System outage", "IT failure"],
    "Execution & Process": ["Data entry error", "Process failure"]
}

business_units = [
    "Finance", "Operations", "IT", "HR", "Compliance",
    "Customer Service", "Supply Chain"
]

risk_owners = ["CRO", "CISO", "Head of Ops", "Finance Director", "Compliance Officer"]

# Severity propensity multipliers (higher => more likely severe)
category_severity_mult = {
    "Internal Fraud": 1.25,
    "External Fraud": 1.20,
    "Employment Practices": 0.95,
    "Clients & Products": 1.05,
    "Damage to Assets": 1.10,
    "Business Disruption": 1.15,
    "Execution & Process": 0.90
}

bu_severity_mult = {
    "Finance": 1.15,
    "Operations": 1.05,
    "IT": 1.20,
    "HR": 0.90,
    "Compliance": 0.95,
    "Customer Service": 1.00,
    "Supply Chain": 1.10
}

## Risk Registry Generation

In [3]:
# Create a synthetic risk registry where each risk receives an ID,
# category, event type, owner, and narrative description.

num_risks = 25

risk_registry = []

for i in range(1, num_risks + 1):
    category = np.random.choice(list(risk_categories.keys()))
    sub_event = np.random.choice(risk_categories[category])

    risk_registry.append({
        "risk_id": f"RISK-{i:03d}",
        "risk_category": category,
        "risk_event_type": sub_event,
        "risk_description": fake.sentence(nb_words=10),
        "risk_owner": np.random.choice(risk_owners)
    })

dim_risk = pd.DataFrame(risk_registry)

## Operational Loss Event Generation

In [4]:
# Generate synthetic operational loss events with event dates, loss amounts,
# recovery values, probability scores, and expected losses.

num_events = 5000
start_date = datetime(2019, 1, 1)

events = []

# Structural severity probability boosts
category_boost = {
    "Internal Fraud": 0.06,
    "External Fraud": 0.05,
    "Business Disruption": 0.07,
    "Damage to Assets": 0.04,
    "Clients & Products": 0.03,
    "Employment Practices": -0.03,
    "Execution & Process": -0.04
}

bu_boost = {
    "IT": 0.06,
    "Finance": 0.05,
    "Supply Chain": 0.04,
    "Operations": 0.02,
    "Customer Service": 0.00,
    "Compliance": -0.02,
    "HR": -0.04
}

for i in range(1, num_events + 1):

    risk = dim_risk.sample(1).iloc[0]
    event_date = start_date + timedelta(days=np.random.randint(0, 2000))

    # Pick business unit
    bu = np.random.choice(business_units)

    # ----------------------------
    # STRUCTURAL EXTREME PROBABILITY
    # ----------------------------
    base_extreme_prob = 0.08

    extreme_prob = (
        base_extreme_prob
        + category_boost.get(risk["risk_category"], 0)
        + bu_boost.get(bu, 0)
    )

    extreme_prob = np.clip(extreme_prob, 0.01, 0.35)

    # ----------------------------
    # DRAW LOSS
    # ----------------------------
    rand = np.random.rand()

    if rand < extreme_prob:
        gross_loss = np.random.lognormal(mean=13.5, sigma=1.0)  # extreme
    elif rand < extreme_prob + 0.20:
        gross_loss = np.random.lognormal(mean=11.5, sigma=1.0)  # high
    else:
        gross_loss = np.random.lognormal(mean=9.0, sigma=0.9)   # normal

    recovery = gross_loss * np.random.uniform(0, 0.4)
    net_loss = gross_loss - recovery

    events.append({
        "event_id": f"EVT-{i:05d}",
        "event_date": event_date,
        "risk_id": risk["risk_id"],
        "business_unit": bu,
        "gross_loss": round(gross_loss, 2),
        "recovery_amount": round(recovery, 2),
        "net_loss": round(net_loss, 2),
        "description": fake.text(max_nb_chars=120)
    })

fact_risk_events = pd.DataFrame(events)

# Probability proxy
risk_frequency = fact_risk_events["risk_id"].value_counts(normalize=True)
fact_risk_events["probability_score"] = fact_risk_events["risk_id"].map(risk_frequency)

# Expected loss
fact_risk_events["expected_loss"] = (
    fact_risk_events["probability_score"] * fact_risk_events["net_loss"]
)

## Mitigation Tracking

In [5]:
# Add mitigation tracking fields to simulate remediation status and due dates.

mitigation_status = ["Not Started", "In Progress", "Completed"]

fact_risk_events["mitigation_status"] = np.random.choice(
    mitigation_status, size=len(fact_risk_events), p=[0.3, 0.45, 0.25]
)

fact_risk_events["mitigation_due_date"] = (
    fact_risk_events["event_date"] + pd.to_timedelta(np.random.randint(30, 180), unit="D")
)

## Exports for Power BI

In [6]:
# Export synthetic dimension and fact tables for optional Power BI testing.

dim_risk.to_csv("../data/processed/dim_risk.csv", index=False)
fact_risk_events.to_csv("../data/processed/fact_risk_events.csv", index=False)